In [41]:
import os
import json
import random
import torch
import numpy as np
import pandas as pd

In [42]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [43]:
base_dir = "/content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M"
preprocessed_dir = os.path.join(base_dir, "AntM2C_preprocessed")
fusion_artifacts_dir = os.path.join(base_dir, "fusion_artifacts")
meta_artifacts_dir = os.path.join(base_dir, "meta_artifacts")
checkpoints_dir = os.path.join(base_dir, "checkpoints")

os.makedirs(checkpoints_dir, exist_ok=True)

print("base_dir:", base_dir)
print("preprocessed_dir:", preprocessed_dir)
print("fusion_artifacts_dir:", fusion_artifacts_dir)
print("meta_artifacts_dir:", meta_artifacts_dir)
print("checkpoints_dir:", checkpoints_dir)

base_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M
preprocessed_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_preprocessed
fusion_artifacts_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/fusion_artifacts
meta_artifacts_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/meta_artifacts
checkpoints_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/checkpoints


In [44]:
# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("Reproducibility settings applied (seed =", SEED, ")")

Reproducibility settings applied (seed = 42 )


In [45]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


In [46]:
# Load configuration files
fusion_config_path = os.path.join(fusion_artifacts_dir, "fusion_config.json")
meta_config_path = os.path.join(meta_artifacts_dir, "meta_config.json")

assert os.path.exists(fusion_config_path), f"Missing fusion_config.json at {fusion_config_path}"
assert os.path.exists(meta_config_path), f"Missing meta_config.json at {meta_config_path}"

with open(fusion_config_path, "r") as f:
    fusion_config = json.load(f)

with open(meta_config_path, "r") as f:
    meta_config = json.load(f)

In [47]:
print("\n--- Loaded fusion_config.json ---")
for k in ["num_users", "num_items", "num_scenes", "d_model", "num_experts", "hidden_dim"]:
    print(f"{k}: {fusion_config[k]}")


--- Loaded fusion_config.json ---
num_users: 67615
num_items: 59142
num_scenes: 5
d_model: 256
num_experts: 4
hidden_dim: 256


In [48]:
print("\n--- Loaded meta_config.json ---")
for k in ["meta_dim", "FEWSHOT_N", "blend_strategy", "cold_rule"]:
    print(f"{k}: {meta_config[k]}")


--- Loaded meta_config.json ---
meta_dim: 256
FEWSHOT_N: 10
blend_strategy: deterministic_alpha
cold_rule: item_only_fewshot


### Load Core Training Artifacts

Here we will loads the precomputed, row-aligned artifacts required for end-to-end CTR training. Interaction-level features (IDs, text embeddings, and time features) must have identical length to guarantee that each row index refers to the same user–item–scene interaction across all modalities. Item-level image embeddings are loaded separately and indexed via item_id_idx.

In [49]:
# paths to artifacts
df_id_path = os.path.join(preprocessed_dir, "split_features", "df_id_encoded.pkl")
text_embeddings_path = os.path.join(preprocessed_dir, "text_embeddings.npy")
time_features_path = os.path.join(preprocessed_dir, "time_features_final.pt")

image_embeddings_path = os.path.join(base_dir, "image_embeddings_vit_base_aligned.pt")
image_missing_mask_path = os.path.join(base_dir, "image_missing_mask.pt")

splits_dir = os.path.join(preprocessed_dir, "splits")
train_idx_path = os.path.join(splits_dir, "train_idx.npy")
val_idx_path   = os.path.join(splits_dir, "val_idx.npy")
test_idx_path  = os.path.join(splits_dir, "test_idx.npy")

In [50]:
# validate file existance
required_paths = {
    "df_id_encoded.pkl": df_id_path,
    "text_embeddings.npy": text_embeddings_path,
    "time_features_final.pt": time_features_path,
    "image_embeddings_vit_base_aligned.pt": image_embeddings_path,
    "image_missing_mask.pt": image_missing_mask_path,
    "train_idx.npy": train_idx_path,
    "val_idx.npy": val_idx_path,
    "test_idx.npy": test_idx_path
}

for name, path in required_paths.items():
    assert os.path.exists(path), f"Missing {name} at: {path}"

print("All required artifacts found.")

All required artifacts found.


In [51]:
# Load interaction-level IDs (row-aligned)
df_id = pd.read_pickle(df_id_path)
print("Loaded df_id_encoded.pkl:", df_id.shape)
print("Columns:", df_id.columns.tolist())
print(df_id.head())

Loaded df_id_encoded.pkl: (3730329, 3)
Columns: ['user_id_idx', 'item_id_idx', 'scene_idx']
   user_id_idx  item_id_idx  scene_idx
0        41688        13590          4
1        39834        21351          2
2        22354        33398          2
3        17547        11190          1
4        55873        33398          2


In [52]:
# Load text embeddings as memmap (interaction-level)
text_embeddings = np.load(text_embeddings_path, mmap_mode="r")
print("Loaded text_embeddings.npy (memmap):", text_embeddings.shape, "| dtype:", text_embeddings.dtype)

Loaded text_embeddings.npy (memmap): (3730329, 1536) | dtype: float32


In [53]:
# Load time features (interaction-level)
time_features = torch.load(time_features_path, map_location="cpu")
print("Loaded time_features_final.pt:", tuple(time_features.shape), "| dtype:", time_features.dtype)

Loaded time_features_final.pt: (3730329, 5) | dtype: torch.float32


In [54]:
# Load image embeddingd + missing mask (item-level)
image_embeddings = torch.load(image_embeddings_path, map_location="cpu")
image_missing_mask = torch.load(image_missing_mask_path, map_location="cpu")

print("Loaded image_embeddings:", tuple(image_embeddings.shape), "| dtype:", image_embeddings.dtype)
print("Loaded image_missing_mask:", tuple(image_missing_mask.shape), "| missing count:", int(image_missing_mask.sum().item()))

Loaded image_embeddings: (59142, 768) | dtype: torch.float32
Loaded image_missing_mask: (59142,) | missing count: 2


In [55]:
# Load deterministic splits
train_idx = np.load(train_idx_path)
val_idx   = np.load(val_idx_path)
test_idx  = np.load(test_idx_path)

print("Loaded splits:")
print("train:", len(train_idx), "val:", len(val_idx), "test:", len(test_idx))

Loaded splits:
train: 2984263 val: 373033 test: 373033


In [56]:
N = len(df_id)

assert text_embeddings.shape[0] == N, f"text_embeddings rows ({text_embeddings.shape[0]}) != df_id rows ({N})"
assert time_features.shape[0] == N, f"time_features rows ({time_features.shape[0]}) != df_id rows ({N})"

EXPECTED_N = 3_730_329
assert N == EXPECTED_N, f"Unexpected df_id row count: {N} (expected {EXPECTED_N})"

# Split indices bounds check
assert train_idx.max() < N and val_idx.max() < N and test_idx.max() < N, "Split indices exceed dataset length"

print("All artifacts loaded and row-alignment verified.")
print(f"Interaction rows N = {N:,} (expected {EXPECTED_N:,})")

All artifacts loaded and row-alignment verified.
Interaction rows N = 3,730,329 (expected 3,730,329)
